# 02 · Modelado con validación por paciente

| | |
|---|---|
| **Objetivo** | Medir cuánto se puede predecir la gravedad (`total_UPDRS`) a partir de la voz, evaluando como en el uso real |
| **Pregunta guía** | ¿El modelo funciona en pacientes que nunca vio? ¿Y para seguir la evolución de un paciente conocido? |
| **Punto de partida** | Notebook 01: el 93% de la varianza de la UPDRS es *entre* pacientes → riesgo alto de fuga de información |
| **Métricas** | **MAE** (error promedio en puntos de UPDRS, fácil de interpretar) y **R²** (0 = igual que predecir el promedio; negativo = peor que eso) |

**Índice**
0. Configuración
1. Resumen ejecutivo
2. Experimento 1 · Fuga de información: partición aleatoria vs. por paciente
3. Experimento 2 · ¿Qué aprende el modelo con partición aleatoria?
4. Experimento 3 · Predecir el nivel de gravedad de un paciente nuevo
5. Experimento 4 · Predecir el cambio respecto de la línea de base del paciente
6. Conclusiones e implicancias

## 0. Configuración

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

from src.data import load_data
from src.modeling import (correr_experimentos, metricas_clave_02, hallazgos_clave_02,
                          particion_aleatoria, particion_por_paciente)
from src.viz import guardar, tarjetas_kpi

pd.set_option("display.float_format", "{:.3f}".format)
OBJETIVO = "total_UPDRS"
df = load_data()

Todos los experimentos se corren de una vez (tarda 1-2 minutos) y los resultados quedan en el diccionario `res`.
El código de cada experimento está en `src/modeling.py`, documentado función por función.

In [2]:
res = correr_experimentos(df, OBJETIVO)

1/4 Fuga de información...
2/4 Nivel de gravedad en pacientes nuevos...
3/4 Cambio respecto de la línea de base...
4/4 Importancia de variables...
Listo.


## 1. Resumen ejecutivo

In [3]:
m = metricas_clave_02(res)

kpis = tarjetas_kpi([
    ("R² · partición aleatoria", m["r2_aleatoria"], ".2f"),
    ("R² · pacientes nuevos", m["r2_paciente"], ".2f"),
    ("MAE · paciente nuevo (referencia)", m["mae_referencia"], ".1f"),
    ("MAE · con línea de base del paciente", m["mejor_cambio_mae"], ".1f"),
])
guardar(kpis, "02_kpis")
kpis.show()

display(Markdown("### Hallazgos clave\n\n" +
                 "\n".join(f"{i}. {h}" for i, h in enumerate(hallazgos_clave_02(m), 1))))

### Hallazgos clave

1. **La partición aleatoria engaña:** el mismo Random Forest obtiene R² = 0.91 (MAE 2.3 puntos) con registros mezclados, pero R² = -0.64 (MAE 11.3) cuando se evalúa en pacientes que no vio. La diferencia es pura fuga de información.
2. **Para un paciente nuevo, ningún modelo supera de forma relevante a predecir el promedio** (MAE de referencia 8.9 puntos; el menor error (8.6) lo obtiene justamente predecir el promedio). Con 42 pacientes, la voz no alcanza para estimar el nivel de gravedad de alguien desconocido.
3. **Con partición aleatoria el modelo 'memoriza pacientes':** la variable más importante es `age` (75% de la importancia total), y la importancia de cada variable se correlaciona con qué tan propia del paciente es (Spearman +0.69). Aprende a reconocer quién es, no qué predice la gravedad.
4. **Conocer la línea de base del paciente es lo que más ayuda:** prediciendo el cambio respecto de sus primeras 2 semanas, el error baja a 3.4 puntos (62% menos que el modelo de referencia para pacientes nuevos). Los cambios en la voz no mejoran a una regla simple sin voz (MAE 3.6 vs. 3.4).

## 2. Experimento 1 · Fuga de información

**Qué hacemos:** entrenamos **el mismo Random Forest con los mismos datos** dos veces. Lo único que cambia es cómo se dividen train y test:

| Partición | Cómo divide | Qué simula |
|---|---|---|
| `KFold(shuffle=True)` | Mezcla todas las grabaciones al azar | Nada realista: el modelo ya "conoce" a cada paciente de test |
| `GroupKFold` | Cada paciente queda entero en train **o** en test | Usar el modelo con un paciente nuevo |

In [4]:
# Así se ve la diferencia en código: GroupKFold recibe los grupos (pacientes)
cv_aleatoria = particion_aleatoria()
cv_paciente = particion_por_paciente()

train_idx, test_idx = next(cv_paciente.split(df, groups=df["subject_id"]))
compartidos = set(df.loc[train_idx, "subject_id"]) & set(df.loc[test_idx, "subject_id"])
print(f"GroupKFold → pacientes en train y test a la vez: {len(compartidos)}")

train_idx, test_idx = next(cv_aleatoria.split(df))
compartidos = set(df.loc[train_idx, "subject_id"]) & set(df.loc[test_idx, "subject_id"])
print(f"KFold aleatorio → pacientes en train y test a la vez: {len(compartidos)}")

GroupKFold → pacientes en train y test a la vez: 0
KFold aleatorio → pacientes en train y test a la vez: 42


In [5]:
res["fuga"]

,particion,MAE,R2
0,Aleatoria (registros mezclados),2.296,0.911
1,Por paciente (GroupKFold),11.251,-0.640


In [6]:
fig = px.scatter(res["predicciones_fuga"], x="real", y="predicho", facet_col="particion",
                 opacity=0.25, render_mode="webgl",
                 title=f"{OBJETIVO}: valor real vs. predicho según la partición",
                 labels={"real": "UPDRS real", "predicho": "UPDRS predicho"})
for col in (1, 2):
    fig.add_shape(type="line", x0=5, y0=5, x1=55, y1=55, line={"dash": "dash", "color": "gray"}, row=1, col=col)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_traces(marker_size=3)
guardar(fig, "02_fuga_real_vs_predicho")
fig

**Cómo leerlo:** con partición aleatoria los puntos se pegan a la diagonal (predicción casi perfecta).
Con pacientes nuevos se dispersan por completo. El modelo no aprendió a predecir la gravedad: aprendió a **reconocer pacientes**.

## 3. Experimento 2 · ¿Qué aprende el modelo con partición aleatoria?

**Hipótesis:** si el modelo memoriza pacientes, sus variables más importantes deberían ser las que mejor
**identifican a cada paciente**, es decir, las que casi no cambian dentro de una misma persona
(alta proporción de varianza *entre* pacientes, calculada en el notebook 01).

Usamos **importancia por permutación**: cuánto empeora el modelo si desordenamos una variable al azar.

In [7]:
res["importancia"]

,importancia,varianza_entre_pacientes
variable,,
age,1.405,1.000
sex,0.207,1.000
DFA,0.175,0.769
Jitter(Abs),0.037,0.431
HNR,0.020,0.606
RPDE,0.013,0.502
PPE,0.007,0.497
Shimmer:APQ11,0.006,0.527
NHR,0.005,0.562


In [8]:
imp = res["importancia"].reset_index()
fig = px.scatter(imp, x="varianza_entre_pacientes", y="importancia", text="variable",
                 title="Importancia en el modelo vs. qué tan 'propia del paciente' es la variable",
                 labels={"varianza_entre_pacientes": "Proporción de varianza entre pacientes",
                         "importancia": "Importancia por permutación"})
fig.update_traces(textposition="top center", marker_size=9)
fig.update_layout(xaxis_tickformat=".0%", height=550)
guardar(fig, "02_importancia_vs_firma")
fig

**Cómo leerlo:** la edad y el sexo son constantes en cada paciente (100% de varianza entre pacientes).
Con solo 42 pacientes, la combinación de edad + sexo + una o dos medidas de voz estables (como `DFA`)
funciona casi como un **documento de identidad**. El modelo los usa para saber de quién es la grabación y devolver su UPDRS.

## 4. Experimento 3 · Nivel de gravedad en un paciente nuevo

**Pregunta:** si llega un paciente que el modelo nunca vio, ¿puede estimar su UPDRS a partir de la voz?

Todos los modelos se evalúan con `GroupKFold`. Comparamos contra **modelos de referencia** (*baselines*):
un modelo solo vale si supera a "predecir el promedio" y a "usar solo edad y sexo".
También probamos **promediar las grabaciones por semana** para reducir el ruido de la voz.

In [9]:
res["nivel"].pivot(index="modelo", columns="datos", values=["MAE", "R2"])

MAE                   \
datos                                 Cada grabación Promedio semanal   
modelo                                                                  
Edad + sexo · Ridge                            8.906            8.764   
Media (sin información)                        8.908            8.648   
Voz + edad + sexo · Gradient Boosting         10.404           10.149   
Voz + edad + sexo · Random Forest             11.251           10.588   
Voz + edad + sexo · Ridge                      9.361           10.164   

                                                  R2                   
datos                                 Cada grabación Promedio semanal  
modelo                                                                 
Edad + sexo · Ridge                            0.009            0.025  
Media (sin información)                       -0.050           -0.009  
Voz + edad + sexo · Gradient Boosting         -0.394           -0.288  
Voz + edad + sexo · Random Forest             -0.640           -0.351  
Voz + edad + sexo · Ridge                     -0.120           -0.350

In [10]:
fig = px.bar(res["nivel"], y="modelo", x="MAE", color="datos", barmode="group", orientation="h",
             title="Error en pacientes nuevos (MAE, menor es mejor)",
             labels={"MAE": "MAE (puntos de UPDRS)", "modelo": ""}, text_auto=".1f")
ref = res["nivel"].query("modelo == 'Media (sin información)' and datos == 'Cada grabación'")["MAE"].iloc[0]
fig.add_vline(x=ref, line_dash="dash", line_color="gray", annotation_text="Predecir el promedio")
fig.update_layout(yaxis={"autorange": "reversed"}, height=450)
guardar(fig, "02_nivel_pacientes_nuevos")
fig

## 5. Experimento 4 · Cambio respecto de la línea de base del paciente

**Cambio de enfoque:** en la práctica clínica el paciente **ya es conocido**. Usamos sus **primeras 2 semanas** como línea de base y preguntamos:
*"¿cuánto cambió su UPDRS desde entonces?"*. Es la pregunta longitudinal del proyecto real.

- **Objetivo:** Δ UPDRS = UPDRS actual − promedio de UPDRS en su línea de base.
- **Predictores:** Δ de cada medida de voz respecto de su propia línea de base + días transcurridos.
- **Referencias:** "no cambió nada", "cambio promedio" y "tendencia lineal en el tiempo".

In [11]:
res["cambio"]

,modelo,MAE,R2
0,Sin cambio (Δ = 0),3.948,-0.220
1,Cambio promedio,3.517,-0.021
2,Tendencia temporal · Lineal,3.403,-0.010
3,Δ voz + días · Ridge,3.588,-0.105
4,Δ voz + días + edad + sexo · Random Forest,4.118,-0.574


In [12]:
fig = px.bar(res["cambio"], y="modelo", x="MAE", orientation="h", text_auto=".2f",
             title="Error al predecir el cambio de UPDRS respecto de la línea de base (MAE)",
             labels={"MAE": "MAE (puntos de UPDRS)", "modelo": ""},
             color=res["cambio"]["modelo"].str.startswith("Δ voz").map({True: "Con voz", False: "Sin voz"}))
fig.update_layout(yaxis={"autorange": "reversed"}, legend_title_text="", height=400)
guardar(fig, "02_cambio_linea_base")
fig

**Cómo leerlo:** conocer la línea de base del paciente reduce el error de unos 9 a unos 3,5 puntos.
Pero los cambios en la voz no mejoran a una simple tendencia temporal. Recordá que la UPDRS intermedia está
**interpolada linealmente** (notebook 01): por construcción, una recta en el tiempo es casi imposible de superar en este dataset.

## 6. Conclusiones e implicancias

**Lo que encontré**
1. **La partición aleatoria infla las métricas de forma dramática.** El mismo Random Forest pasa de R² = 0,91 (MAE 2,3 puntos) a R² = −0,64 (MAE 11,3) cuando se evalúa en pacientes que no vio. Es la misma lección que en el proyecto de Vaca Muerta, pero mucho más extrema, porque el 93% de la varianza es entre pacientes.
2. **El modelo con partición aleatoria memoriza pacientes.** La edad concentra cerca del 75% de la importancia, y la importancia de cada variable se correlaciona con qué tan propia del paciente es (Spearman ≈ +0,69). Edad + sexo + `DFA` funcionan como un identificador.
3. **Para pacientes nuevos, ningún modelo supera a predecir el promedio** (MAE ≈ 8,6–8,9 puntos). Los modelos más flexibles (Random Forest, Gradient Boosting) son incluso peores, porque sobreajustan a los 33-34 pacientes de entrenamiento. Promediar por semana mejora apenas.
4. **Lo que más reduce el error es conocer la línea de base del paciente:** el MAE baja a ≈ 3,4 puntos. Pero los cambios en la voz no aportan sobre una tendencia temporal simple.

**Limitaciones**
- UPDRS interpolada: favorece artificialmente a la tendencia lineal y limita lo que la voz puede aportar dentro del paciente.
- 42 pacientes: con 5 folds, cada modelo se entrena con ~34 pacientes. Es poco para modelos flexibles.
- Una sola partición en folds: con más tiempo, repetir la validación con distintas semillas para medir la estabilidad.

**Implicancias para un proyecto real (Parkinson y termorregulación)**
- Validar **siempre por paciente** y reportar contra baselines simples. Un R² alto con partición aleatoria no dice nada.
- Encuadrar la pregunta como **longitudinal dentro del paciente** (línea de base + cambio, modelos mixtos, análisis de supervivencia) en lugar de "predecir el valor de cualquier registro".
- Revisar cómo se midió el desenlace (¿evaluación clínica real o interpolada?, ¿cada cuánto?) antes de modelar.

**Próximo paso:** notebook 03. Modelos mixtos (efecto aleatorio por paciente) y un LSTM simple sobre secuencias semanales, para comparar con estos baselines.